## 
The cleaned sales data was grouped by transaction date, and the quantities sold were summed to calculate total daily demand. The transaction dates were converted to datetime format and sorted chronologically to prepare the dataset for time-series forecasting.

## 
The resulting daily_total dataset contains two columns: transaction_date and daily_quantity. This dataset provides the daily demand values needed to establish a baseline, train forecasting models, and evaluate their performance.

In [4]:
# Convert transaction_date to datetime
sales["transaction_date"] = pd.to_datetime(
    sales["transaction_date"]
)

# Aggregate quantity by day
daily_total = (
    sales.groupby("transaction_date", as_index=False)
    ["quantity"]
    .sum()
    .rename(columns={"quantity": "daily_quantity"})
    .sort_values("transaction_date")
)

# Check the forecasting dataset
print("Daily demand shape:", daily_total.shape)
print("Columns:", daily_total.columns.tolist())

print(
    "Date range:",
    daily_total["transaction_date"].min(),
    "to",
    daily_total["transaction_date"].max()
)

print("Missing values:", daily_total.isna().sum().to_dict())

daily_total.head()

Daily demand shape: (731, 2)
Columns: ['transaction_date', 'daily_quantity']
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Missing values: {'transaction_date': 0, 'daily_quantity': 0}


,transaction_date,daily_quantity
0,2024-01-01,5209
1,2024-01-02,4302
2,2024-01-03,4279
3,2024-01-04,4408
4,2024-01-05,5095


#### The daily demand data was sorted chronologically, and a weekly seasonal-naive baseline was created using demand from seven days earlier to predict the current day's demand. This approach assumes that demand follows a recurring weekly pattern.
#### The first seven days have missing baseline forecasts because no previous-week values are available. This baseline provides a simple benchmark for evaluating whether the Random Forest model improves forecasting accuracy.

In [5]:
# Sort the daily demand data by date
forecast_data = daily_total.copy()

forecast_data["transaction_date"] = pd.to_datetime(
    forecast_data["transaction_date"]
)

forecast_data = forecast_data.sort_values(
    "transaction_date"
).reset_index(drop=True)

# Baseline: use demand from 7 days earlier
forecast_data["baseline_forecast"] = (
    forecast_data["daily_quantity"].shift(7)
)

# Check the baseline
print("Dataset shape:", forecast_data.shape)

print(
    forecast_data[
        ["transaction_date", "daily_quantity", "baseline_forecast"]
    ].head(12)
)

print(
    "Rows without a baseline forecast:",
    forecast_data["baseline_forecast"].isna().sum()
)

Dataset shape: (731, 3)
   transaction_date  daily_quantity  baseline_forecast
0        2024-01-01            5209                NaN
1        2024-01-02            4302                NaN
2        2024-01-03            4279                NaN
3        2024-01-04            4408                NaN
4        2024-01-05            5095                NaN
5        2024-01-06            5811                NaN
6        2024-01-07            5516                NaN
7        2024-01-08            4426             5209.0
8        2024-01-09            4393             4302.0
9        2024-01-10            4350             4279.0
10       2024-01-11            4196             4408.0
11       2024-01-12            5231             5095.0
Rows without a baseline forecast: 7


#### The daily demand data was sorted chronologically and divided into training and testing sets, with 90% of the observations used for training and the final 10% reserved for testing. This resulted in 657 training days and 74 testing days.
-  The training period runs from 1 January 2024 to 18 October 2025, while the testing period runs from 19 October 2025 to 31 December 2025. This time-aware split ensures that the model is evaluated on later dates than those used for training, reflecting a realistic forecasting scenario.

In [6]:
# Sort data chronologically
forecast_data = forecast_data.sort_values(
    "transaction_date"
).reset_index(drop=True)

# Split: first 90% for training, last 10% for testing
split_index = int(len(forecast_data) * 0.9)

train = forecast_data.iloc[:split_index].copy()
test = forecast_data.iloc[split_index:].copy()

# Check the split
print("Training rows:", len(train))
print("Testing rows:", len(test))

print(
    "Training period:",
    train["transaction_date"].min().date(),
    "to",
    train["transaction_date"].max().date()
)

print(
    "Testing period:",
    test["transaction_date"].min().date(),
    "to",
    test["transaction_date"].max().date()
)

Training rows: 657
Testing rows: 74
Training period: 2024-01-01 to 2025-10-18
Testing period: 2025-10-19 to 2025-12-31


In [7]:
from pathlib import Path

downloads = Path.home() / "Downloads"

external_files = [
    "kenya_county_population_2019.csv",
    "table4_core_inflation.csv",
    "kenya_public_holidays.csv"
]

for filename in external_files:
    path = downloads / filename
    print(filename, "—", "Found" if path.exists() else "Not found")

kenya_county_population_2019.csv — Found
table4_core_inflation.csv — Found
kenya_public_holidays.csv — Found


#### Loading the Datasets Required

In [8]:
import pandas as pd
from pathlib import Path

downloads = Path.home() / "Downloads"

# Load external Kenyan datasets
population = pd.read_csv(
    downloads / "kenya_county_population_2019.csv"
)

inflation = pd.read_csv(
    downloads / "table4_core_inflation.csv"
)

holidays = pd.read_csv(
    downloads / "kenya_public_holidays.csv"
)

# Inspect each dataset
external_datasets = {
    "Population": population,
    "Core inflation": inflation,
    "Public holidays": holidays
}

for name, df in external_datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head())


Population
Shape: (5, 2)
Columns: ['county', 'population_2019']
    county  population_2019
0  Nairobi          4397073
1   Kiambu          2417735
2   Nakuru          2162202
3  Mombasa          1208333
4   Kisumu          1155574

Core inflation
Shape: (13, 5)
Columns: ['Period', 'Core Index', 'Core Inflation', 'Non-Core Index', 'Non-Core Inflation']
   Period  Core Index  Core Inflation  Non-Core Index  Non-Core Inflation
0  Dec-24      127.50             2.2          198.51                 4.9
1  Jan-25      127.77             2.0          202.87                 6.8
2  Feb-25      128.01             1.9          204.01                 7.7
3  Mar-25      128.35             2.2          205.55                 7.4
4  Apr-25      128.48             2.5          206.86                 8.4

Public holidays
Shape: (27, 2)
Columns: ['date', 'holiday_name']
         date    holiday_name
0  2024-01-01  New Year's Day
1  2024-03-29     Good Friday
2  2024-04-01   Easter Monday
3  2024-04-10 

#### calendar features were created from the transaction dates to help the forecasting model identify possible patterns in daily demand. These features include the day of the week, month, day of the month, and a public holiday indicator based on the Kenyan public holidays dataset.
- A value of 1 indicates a public holiday, while 0 indicates a non-holiday. The dataset matched 27 public holiday dates. These features were prepared as potential predictors for the Random Forest model to evaluate whether calendar patterns help explain variations in daily demand.

In [9]:
# Convert dates to datetime
forecast_data["transaction_date"] = pd.to_datetime(
    forecast_data["transaction_date"]
)
holidays["date"] = pd.to_datetime(holidays["date"])

# Create a holiday indicator
holiday_dates = holidays["date"].drop_duplicates()

forecast_data["is_public_holiday"] = (
    forecast_data["transaction_date"].isin(holiday_dates).astype(int)
)

# Add basic calendar features
forecast_data["day_of_week"] = (
    forecast_data["transaction_date"].dt.dayofweek
)

forecast_data["month"] = (
    forecast_data["transaction_date"].dt.month
)

forecast_data["day_of_month"] = (
    forecast_data["transaction_date"].dt.day
)

# Check the new features
print(
    forecast_data[
        [
            "transaction_date",
            "daily_quantity",
            "is_public_holiday",
            "day_of_week",
            "month",
            "day_of_month"
        ]
    ].head(10)
)

print(
    "Public holidays matched:",
    forecast_data["is_public_holiday"].sum()
)

  transaction_date  daily_quantity  is_public_holiday  day_of_week  month  \
0       2024-01-01            5209                  1            0      1   
1       2024-01-02            4302                  0            1      1   
2       2024-01-03            4279                  0            2      1   
3       2024-01-04            4408                  0            3      1   
4       2024-01-05            5095                  0            4      1   
5       2024-01-06            5811                  0            5      1   
6       2024-01-07            5516                  0            6      1   
7       2024-01-08            4426                  0            0      1   
8       2024-01-09            4393                  0            1      1   
9       2024-01-10            4350                  0            2      1   

   day_of_month  
0             1  
1             2  
2             3  
3             4  
4             5  
5             6  
6             7  
7       

#### The Kenyan core inflation dataset was prepared by converting its monthly period column into datetime format and renaming the core inflation column for clarity. A monthly key was then created in both the inflation and daily demand datasets to merge the economic data with daily demand.
- The merge identified 335 rows with missing inflation values because the available inflation data begins in December 2024, while the demand dataset begins in January 2024. These missing values were left unchanged rather than filled with invented figures. Core inflation was therefore excluded from the initial Random Forest model to allow evaluation across the full dataset.

In [10]:
# Prepare the monthly core inflation data
inflation["Period"] = pd.to_datetime(
    inflation["Period"],
    format="%b-%y"
)

inflation = inflation.rename(
    columns={
        "Period": "month_date",
        "Core Inflation": "core_inflation_pct"
    }
)

# Create a month key in both datasets
forecast_data["month_date"] = (
    forecast_data["transaction_date"].dt.to_period("M")
)

inflation["month_date"] = (
    inflation["month_date"].dt.to_period("M")
)

# Merge monthly inflation into daily demand
forecast_data = forecast_data.merge(
    inflation[["month_date", "core_inflation_pct"]],
    on="month_date",
    how="left"
)

# Check coverage
print(
    "Rows with missing core inflation:",
    forecast_data["core_inflation_pct"].isna().sum()
)

print(
    forecast_data[
        ["transaction_date", "daily_quantity", "core_inflation_pct"]
    ].head(10)
)

Rows with missing core inflation: 335
  transaction_date  daily_quantity  core_inflation_pct
0       2024-01-01            5209                 NaN
1       2024-01-02            4302                 NaN
2       2024-01-03            4279                 NaN
3       2024-01-04            4408                 NaN
4       2024-01-05            5095                 NaN
5       2024-01-06            5811                 NaN
6       2024-01-07            5516                 NaN
7       2024-01-08            4426                 NaN
8       2024-01-09            4393                 NaN
9       2024-01-10            4350                 NaN


### Preparing Features for the Forecasting Model
- Four calendar-based features were selected as predictors: day of the week, month, day of the month, and public holiday indicator. The daily quantity sold was set as the target variable that the model aims to predict.

- The data was divided chronologically into 657 training observations and 74 testing observations. Both datasets contained four features with no missing values, making them ready for training and evaluation of the Random Forest model.

In [11]:
# Select the features for the first forecasting model
feature_columns = [
    "day_of_week",
    "month",
    "day_of_month",
    "is_public_holiday"
]

# Create chronological training and testing datasets
split_index = int(len(forecast_data) * 0.9)

train = forecast_data.iloc[:split_index].copy()
test = forecast_data.iloc[split_index:].copy()

X_train = train[feature_columns]
y_train = train["daily_quantity"]

X_test = test[feature_columns]
y_test = test["daily_quantity"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Missing training features:", X_train.isna().sum().sum())
print("Missing testing features:", X_test.isna().sum().sum())

Training features: (657, 4)
Testing features: (74, 4)
Missing training features: 0
Missing testing features: 0


## Training model using RandomForestRegressor

In [12]:
from sklearn.ensemble import RandomForestRegressor

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train using historical data only
rf_model.fit(X_train, y_train)

# Predict demand for the test period
test = test.copy()
test["rf_forecast"] = rf_model.predict(X_test)

print("Model training complete.")
print("Test predictions:", len(test))

print(
    test[
        ["transaction_date", "daily_quantity", "rf_forecast"]
    ].head(10)
)

Model training complete.
Test predictions: 74
    transaction_date  daily_quantity  rf_forecast
657       2025-10-19            7681      7292.99
658       2025-10-20            7397      5692.06
659       2025-10-21            6230      5484.87
660       2025-10-22            6483      5555.66
661       2025-10-23            6256      5587.50
662       2025-10-24            7108      6355.90
663       2025-10-25            8411      7563.34
664       2025-10-26            7983      7002.38
665       2025-10-27            6069      5565.63
666       2025-10-28            6167      5389.76


#### Generating Baseline Forecasts for the Test Period
- A lookup table was created using actual daily demand indexed by transaction date. For each test date, the baseline forecast was obtained from the actual demand recorded exactly seven days earlier in the full daily demand dataset.

- This approach ensures that the baseline has predictions for the entire 74-day test period, including its first seven days. The missing-prediction check was used to verify that every test date had a corresponding baseline forecast.

In [15]:
# Create a lookup of actual daily demand by date
actual_by_date = daily_total.set_index(
    "transaction_date"
)["daily_quantity"]

# Ensure dates are in datetime format
test["transaction_date"] = pd.to_datetime(
    test["transaction_date"]
)
daily_total["transaction_date"] = pd.to_datetime(
    daily_total["transaction_date"]
)

# Predict demand using the actual demand from 7 days earlier
test["baseline_forecast"] = (
    test["transaction_date"] - pd.Timedelta(days=7)
).map(actual_by_date)

# Check for missing baseline predictions
print(
    "Missing baseline predictions:",
    test["baseline_forecast"].isna().sum()
)

Missing baseline predictions: 0


#### Evaluating and Comparing the Models
- The weekly seasonal-naive baseline and Random Forest model were evaluated on the same 74 test days using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

- The baseline achieved an MAE of 467.47 and an RMSE of 692.09, while Random Forest recorded an MAE of 1,068.59 and an RMSE of 1,143.11. Since lower values indicate better predictive accuracy, the weekly baseline outperformed Random Forest on both metrics and was the preferred model for this evaluation period.

In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = pd.DataFrame({
    "Model": [
        "Weekly seasonal-naive baseline",
        "Random Forest"
    ],
    "MAE": [
        mean_absolute_error(
            test["daily_quantity"],
            test["baseline_forecast"]
        ),
        mean_absolute_error(
            test["daily_quantity"],
            test["rf_forecast"]
        )
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(
            test["daily_quantity"],
            test["baseline_forecast"]
        )),
        np.sqrt(mean_squared_error(
            test["daily_quantity"],
            test["rf_forecast"]
        ))
    ]
})

print(results.round(2))

                            Model      MAE     RMSE
0  Weekly seasonal-naive baseline   467.47   692.09
1                   Random Forest  1068.59  1143.11


#### Random Forest Error Analysis
- Prediction errors were calculated by subtracting the Random Forest forecasts from actual daily demand. The absolute error was then calculated to measure the size of each mistake, regardless of whether demand was overestimated or underestimated.

- The 10 dates with the largest absolute errors were identified to investigate where the model performed poorly. The results showed substantial underestimation on several high-demand days, particularly in November and December 2025. This suggests that the calendar-based features alone did not fully capture changes in demand levels.

In [18]:
# Calculate prediction errors
test["rf_error"] = (
    test["daily_quantity"] - test["rf_forecast"]
)

test["rf_absolute_error"] = test["rf_error"].abs()

# Show the 10 dates with the largest Random Forest errors
largest_errors = test.nlargest(
    10, "rf_absolute_error"
)[
    [
        "transaction_date",
        "daily_quantity",
        "rf_forecast",
        "rf_error",
        "rf_absolute_error"
    ]
]

print(largest_errors.round(2).to_string(index=False))

transaction_date  daily_quantity  rf_forecast  rf_error  rf_absolute_error
      2025-12-26           10855      8138.62   2716.38            2716.38
      2025-12-12           10526      8171.21   2354.79            2354.79
      2025-11-23            9149      7365.24   1783.76            1783.76
      2025-10-20            7397      5692.06   1704.94            1704.94
      2025-12-25            9789      8101.22   1687.78            1687.78
      2025-11-22            9463      7795.53   1667.47            1667.47
      2025-11-01            9481      7854.81   1626.19            1626.19
      2025-12-07           10595      9046.15   1548.85            1548.85
      2025-11-02            8602      7113.67   1488.33            1488.33
      2025-11-08            9126      7649.70   1476.30            1476.30


C:\Users\HP\AppData\Local\Temp\ipykernel_14900\2268636382.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(largest_errors.round(2).to_string(index=False))


## Model Errors

In [19]:
# Calculate baseline errors
test["baseline_error"] = (
    test["daily_quantity"] - test["baseline_forecast"]
)

test["baseline_absolute_error"] = (
    test["baseline_error"].abs()
)

# Compare both models on the 10 worst Random Forest dates
comparison = test.nlargest(
    10, "rf_absolute_error"
)[
    [
        "transaction_date",
        "daily_quantity",
        "baseline_forecast",
        "baseline_absolute_error",
        "rf_forecast",
        "rf_absolute_error"
    ]
].copy()

print(comparison.to_string(index=False))

transaction_date  daily_quantity  baseline_forecast  baseline_absolute_error  rf_forecast  rf_absolute_error
      2025-12-26           10855               9028                     1827      8138.62            2716.38
      2025-12-12           10526               9143                     1383      8171.21            2354.79
      2025-11-23            9149               8470                      679      7365.24            1783.76
      2025-10-20            7397               6114                     1283      5692.06            1704.94
      2025-12-25            9789               8294                     1495      8101.22            1687.78
      2025-11-22            9463               9237                      226      7795.53            1667.47
      2025-11-01            9481               8411                     1070      7854.81            1626.19
      2025-12-07           10595               8369                     2226      9046.15            1548.85
      2025-11-02   

In [20]:
# Compare average absolute errors by month
test["month_name"] = test["transaction_date"].dt.strftime("%B")

monthly_errors = (
    test.groupby("month_name")
    .agg(
        actual_demand=("daily_quantity", "mean"),
        baseline_MAE=("baseline_absolute_error", "mean"),
        random_forest_MAE=("rf_absolute_error", "mean")
    )
    .round(2)
)

print(monthly_errors)

            actual_demand  baseline_MAE  random_forest_MAE
month_name                                                
December          8973.58        632.71            1202.10
November          7527.43        331.10            1029.87
October           6892.69        388.15             839.56


## Conclusion on Model Errors
Both models make forecasting errors, but the weekly seasonal-naive baseline is more accurate. Random Forest frequently underestimates high-demand days, particularly in November and December. The baseline also misses some demand peaks, so neither model predicts demand perfectly.

## 
The weekly seasonal-naive baseline performed better than Random Forest, achieving a lower MAE (467.47 vs. 1,068.59) and RMSE (692.09 vs. 1,143.11). Therefore, the baseline is the preferred model for this evaluation period. Future improvements could include recent demand patterns and additional relevant features to improve forecast accuracy.